In [ ]:
import os, sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.chdir(os.path.expanduser("C:\\Users\\naqee\\OneDrive\\Desktop\\CSC415 Project\\RAEDiTRobotics"))  # adjust for your machine
sys.path.insert(0, os.getcwd())

import torch
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt

from data_pipeline.datasets.stage1_dataset import Stage1Dataset
from models.encoder import FrozenMultiViewEncoder
from models.adapter import TrainableAdapter
from models.decoder import ViTDecoder
from models.discriminator import PatchDiscriminator
from models.losses import create_lpips_net, l1_loss, lpips_loss_fn
from training.train_stage1 import load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
CHECKPOINT = "checkpoints/stage1-full/epoch_007.pt"

encoder = FrozenMultiViewEncoder(pretrained=True).to(device).eval()
adapter = TrainableAdapter().to(device)
decoder = ViTDecoder().to(device)
disc = PatchDiscriminator(pretrained=True).to(device)

# Load checkpoint, stripping torch.compile prefix if present
ckpt = torch.load(CHECKPOINT, weights_only=False, map_location="cpu")

def strip_compile_prefix(state_dict):
    return {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}

adapter.load_state_dict(strip_compile_prefix(ckpt["adapter"]))
decoder.load_state_dict(strip_compile_prefix(ckpt["decoder"]))
disc.head.load_state_dict(strip_compile_prefix(ckpt["disc_head"]))
print(f"Loaded checkpoint from epoch {ckpt['epoch']}")

adapter.eval()
decoder.eval()


In [ ]:
@torch.no_grad()
def reconstruct_sample(sample):
    """Run one sample through encoder→adapter→decoder. Returns (originals, reconstructions, view_mask)."""
    imgs_enc = sample["images_enc"].unsqueeze(0).to(device)
    imgs_tgt = sample["images_target"]
    vp = sample["view_present"]

    B, K = 1, vp.shape[0]
    mask = vp.reshape(-1)
    x = imgs_enc.reshape(B * K, 3, 224, 224)[mask]

    tokens = encoder(x)
    adapted = adapter(tokens)
    pred = decoder(adapted)

    # Scatter reconstructions back to view slots
    originals = []
    reconstructions = []
    idx = 0
    for k in range(K):
        if vp[k]:
            originals.append(imgs_tgt[k].permute(1, 2, 0).cpu().numpy())
            reconstructions.append(pred[idx].cpu().permute(1, 2, 0).numpy())
            idx += 1

    return originals, reconstructions, vp


In [ ]:
# Adjust base path for your machine
DATA_BASE = "C:/Users/naqee/OneDrive/Desktop/CSC415 Project/data/unified"

TASKS = {
    # Robomimic (4 camera views: agentview, eye_in_hand, + 2 placeholders)
    "lift":       f"{DATA_BASE}/robomimic/lift/ph.hdf5",
    "can":        f"{DATA_BASE}/robomimic/can/ph.hdf5",
    "square":     f"{DATA_BASE}/robomimic/square/ph.hdf5",
    # RLBench (1 front camera view)
    "close_jar":  f"{DATA_BASE}/rlbench/close_jar.hdf5",
    "open_drawer": f"{DATA_BASE}/rlbench/open_drawer.hdf5",
    "push_buttons": f"{DATA_BASE}/rlbench/push_buttons.hdf5",
    "turn_tap":   f"{DATA_BASE}/rlbench/turn_tap.hdf5",
    "meat_off_grill": f"{DATA_BASE}/rlbench/meat_off_grill.hdf5",
}

# Verify files exist
for name, path in TASKS.items():
    exists = "OK" if os.path.isfile(path) else "MISSING"
    print(f"  {name:20s} {exists}")


In [ ]:
# Pick samples: use specific indices or None for evenly-spaced defaults
SAMPLE_INDICES = None  # e.g. {"lift": [0, 50, 100], "close_jar": [10, 20]}

for task_name, hdf5_path in TASKS.items():
    if not os.path.isfile(hdf5_path):
        print(f"{task_name}: MISSING")
        continue

    ds = Stage1Dataset(hdf5_path, split="valid")

    if SAMPLE_INDICES and task_name in SAMPLE_INDICES:
        indices = SAMPLE_INDICES[task_name]
    else:
        indices = [len(ds) // 3, 2 * len(ds) // 3]  # default: 1/3 and 2/3 through

    for idx in indices:
        idx = min(idx, len(ds) - 1)
        sample = ds[idx]
        origs, recons, vp = reconstruct_sample(sample)
        n_views = len(origs)

        fig, axes = plt.subplots(2, n_views, figsize=(4 * n_views, 8))
        if n_views == 1:
            axes = axes.reshape(2, 1)
        fig.suptitle(f"{task_name} — sample {idx} / {len(ds)}", fontsize=14)

        for v in range(n_views):
            axes[0, v].imshow(np.clip(origs[v], 0, 1))
            axes[0, v].set_title(f"View {v} original")
            axes[0, v].axis("off")
            axes[1, v].imshow(np.clip(recons[v], 0, 1))
            axes[1, v].set_title(f"View {v} recon")
            axes[1, v].axis("off")

        plt.tight_layout()
        plt.show()


In [ ]:
CHECKPOINT_A = "checkpoints/stage1-full/epoch_000.pt"
CHECKPOINT_B = "checkpoints/stage1-full/epoch_007.pt"
LABEL_A = "multi-task (epoch 0)"
LABEL_B = "multi-task (epoch 7)"

# Pick samples per task: None for evenly-spaced defaults
COMPARE_INDICES = None  # e.g. {"lift": [0, 50], "close_jar": [10, 30]}
N_SAMPLES = 2

# Load checkpoint A
adapter_a = TrainableAdapter().to(device)
decoder_a = ViTDecoder().to(device)
ckpt_a = torch.load(CHECKPOINT_A, weights_only=False, map_location="cpu")
adapter_a.load_state_dict(strip_compile_prefix(ckpt_a["adapter"]))
decoder_a.load_state_dict(strip_compile_prefix(ckpt_a["decoder"]))
adapter_a.eval(); decoder_a.eval()
print(f"Checkpoint A: epoch {ckpt_a['epoch']}")

# Load checkpoint B
adapter_b = TrainableAdapter().to(device)
decoder_b = ViTDecoder().to(device)
ckpt_b = torch.load(CHECKPOINT_B, weights_only=False, map_location="cpu")
adapter_b.load_state_dict(strip_compile_prefix(ckpt_b["adapter"]))
decoder_b.load_state_dict(strip_compile_prefix(ckpt_b["decoder"]))
adapter_b.eval(); decoder_b.eval()
print(f"Checkpoint B: epoch {ckpt_b['epoch']}")

for task_name, hdf5_path in TASKS.items():
    if not os.path.isfile(hdf5_path):
        print(f"{task_name}: MISSING")
        continue

    ds = Stage1Dataset(hdf5_path, split="valid")

    if COMPARE_INDICES and task_name in COMPARE_INDICES:
        indices = COMPARE_INDICES[task_name]
    else:
        indices = [(i + 1) * len(ds) // (N_SAMPLES + 1) for i in range(N_SAMPLES)]

    for idx in indices:
        idx = min(idx, len(ds) - 1)
        sample = ds[idx]
        imgs_enc = sample["images_enc"].unsqueeze(0).to(device)
        imgs_tgt = sample["images_target"]
        vp = sample["view_present"]
        mask = vp.reshape(-1)
        x = imgs_enc.reshape(-1, 3, 224, 224)[mask]

        with torch.no_grad():
            tokens = encoder(x)
            pred_a = decoder_a(adapter_a(tokens))
            pred_b = decoder_b(adapter_b(tokens))

        n_views = int(mask.sum())
        fig, axes = plt.subplots(3, n_views, figsize=(4 * n_views, 12))
        if n_views == 1:
            axes = axes.reshape(3, 1)
        fig.suptitle(f"{task_name} — sample {idx} / {len(ds)}", fontsize=14)

        vi = 0
        for k in range(len(vp)):
            if not vp[k]:
                continue
            orig = imgs_tgt[k].permute(1, 2, 0).numpy()
            rec_a = pred_a[vi].cpu().permute(1, 2, 0).numpy()
            rec_b = pred_b[vi].cpu().permute(1, 2, 0).numpy()

            axes[0, vi].imshow(np.clip(orig, 0, 1))
            axes[0, vi].set_title(f"View {k} original")
            axes[0, vi].axis("off")
            axes[1, vi].imshow(np.clip(rec_a, 0, 1))
            axes[1, vi].set_title(f"{LABEL_A}")
            axes[1, vi].axis("off")
            axes[2, vi].imshow(np.clip(rec_b, 0, 1))
            axes[2, vi].set_title(f"{LABEL_B}")
            axes[2, vi].axis("off")
            vi += 1

        plt.tight_layout()
        plt.show()

In [ ]:
from torch.utils.data import DataLoader

lpips_net = create_lpips_net().to(device)

print(f"{'Task':20s} {'Val L1':>8s} {'Val LPIPS':>10s} {'Val Rec':>9s} {'Samples':>8s}")
print("-" * 60)

for task_name, hdf5_path in TASKS.items():
    if not os.path.isfile(hdf5_path):
        print(f"{task_name:20s} MISSING")
        continue

    ds = Stage1Dataset(hdf5_path, split="valid")
    loader = DataLoader(ds, batch_size=8, num_workers=0)

    total_l1, total_lpips, n = 0.0, 0.0, 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            B, K = batch["view_present"].shape
            mask = batch["view_present"].reshape(-1)
            x = batch["images_enc"].reshape(B * K, 3, 224, 224)[mask]
            tgt = batch["images_target"].reshape(B * K, 3, 224, 224)[mask]

            tokens = encoder(x)
            pred = decoder(adapter(tokens))
            total_l1 += l1_loss(pred, tgt).item()
            total_lpips += lpips_loss_fn(pred, tgt, lpips_net).item()
            n += 1

    l1 = total_l1 / n
    lp = total_lpips / n
    print(f"{task_name:20s} {l1:8.4f} {lp:10.4f} {l1+lp:9.4f} {len(ds):8d}")


In [ ]:
# Find best and worst reconstructions for a specific task
EVAL_TASK = "lift"  # change as needed
ds = Stage1Dataset(TASKS[EVAL_TASK], split="valid")

errors = []
with torch.no_grad():
    for i in range(min(200, len(ds))):
        sample = ds[i]
        origs, recons, _ = reconstruct_sample(sample)
        err = np.mean(np.abs(np.array(origs[0]) - np.array(recons[0])))
        errors.append((i, err))

errors.sort(key=lambda x: x[1])
best_5 = errors[:5]
worst_5 = errors[-5:]

fig, axes = plt.subplots(5, 4, figsize=(16, 20))
fig.suptitle(f"{EVAL_TASK}: Best 5 (left) vs Worst 5 (right)", fontsize=14)

for row, ((bi, be), (wi, we)) in enumerate(zip(best_5, worst_5)):
    for col, (idx, err, label) in enumerate([
        (bi, be, "best orig"), (bi, be, "best recon"),
        (wi, we, "worst orig"), (wi, we, "worst recon"),
    ]):
        sample = ds[idx]
        origs, recons, _ = reconstruct_sample(sample)
        img = origs[0] if "orig" in label else recons[0]
        axes[row, col].imshow(np.clip(img, 0, 1))
        axes[row, col].set_title(f"{label} (L1={err:.4f})", fontsize=9)
        axes[row, col].axis("off")

plt.tight_layout()
plt.savefig(f"eval_{EVAL_TASK}_best_worst.png", dpi=150, bbox_inches="tight")
plt.show()
